# 01 · Adquisición de datos (capa raw)

**Proyecto:** previsión de incidencias en Cercanías Madrid · Actividad 1

Este notebook **solo descarga y guarda** los datos tal y como los publica cada fuente. No filtra, no transforma y no limpia: eso se hará en el notebook de limpieza, que leerá de `data/raw/` y escribirá en `data/silver/`.

| Fuente | Qué guardamos | Formato en disco |
|---|---|---|
| GTFS estático de Renfe Cercanías | El ZIP completo (todos los núcleos), versionado por fecha de descarga y hash | `.zip` |
| GTFS-RT `trip_updates` | Instantáneas del feed (todos los núcleos) | `.pb` (protobuf) |
| GTFS-RT `alerts` | Instantáneas del feed (todos los núcleos) | `.pb` (protobuf) |
| Open-Meteo | Meteorología horaria de Madrid: últimos días y previsión | `.json` |

Cada descarga, correcta o fallida, queda registrada en `data/raw/manifest.csv`. Ese registro es el que nos permitirá distinguir **ausencia de datos** (el colector falló) de **ausencia de incidencias**.

**Decisiones:**

- **Se guarda el `.pb` y no el `.json` de GTFS-RT.** El protobuf es el formato estándar de la especificación y el que se parsea con la librería oficial (`gtfs-realtime-bindings`). La versión JSON de Renfe es una conversión cuyo esquema no está documentado formalmente.
- **No se filtra Madrid aquí.** Filtrar ya es procesar; además, guardar todos los núcleos deja abierta la ampliación prevista en la actividad 0.
- **El GTFS estático se versiona.** Los `trip_id` cambian entre versiones mensuales, así que necesitamos la versión vigente en cada captura de tiempo real para poder cruzarlas.

**Requisitos:** `pip install -r requirements.txt`. Ejecutar desde la raíz del repositorio o desde `notebooks/`.

## 1. Obtenermos inicialmente la respuesta

In [ ]:
import requests
response = requests.get("https://gtfsrt.renfe.com/trip_updates.pb", timeout=5)
print(response.content)

## Leemos la respuesta y la pasmos a texto legible

In [7]:
from google.transit import gtfs_realtime_pb2
from datetime import datetime, timezone



feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

hora_feed = datetime.fromtimestamp(feed.header.timestamp, tz=timezone.utc)

print("Versión GTFS-RT:", feed.header.gtfs_realtime_version)
print("Hora del feed (UTC):", hora_feed)
print("Hora del feed (Madrid):", hora_feed.astimezone())
print("Entidades:", len(feed.entity))

Versión GTFS-RT: 2.0
Hora del feed (UTC): 2026-11-17 18:06:45+00:00
Hora del feed (Madrid): 2026-11-17 19:06:45+01:00
Entidades: 271


## Entidad por dentro

In [8]:
primera = feed.entity[0]
print(primera)

id: "TUUPDATE_3020M23075C2"
trip_update {
  trip {
    trip_id: "3020M23075C2"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    arrival {
      delay: 0
      time: 1794941460
    }
    stop_id: "51051"
  }
  vehicle {
    wheelchair_accessible: WHEELCHAIR_INACCESSIBLE
  }
  delay: 0
}



# Estructura de Campos GTFS-RT (Renfe)

| Campo | Descripción |
| :--- | :--- |
| `id` | Identifica el mensaje dentro del feed, no el tren. Renfe lo forma como prefijo + `trip_id` (`TUUPDATE_` + `3020M23075C2`). Sirve para que no haya dos entidades iguales en un mismo feed; para nosotros no aporta nada que no tenga el `trip_id`. |
| `trip.trip_id` | El viaje programado: un tren concreto en un horario concreto. Es la clave para cruzar con el GTFS estático. |
| `trip.schedule_relationship` | Relación con el horario: `SCHEDULED` (va según lo programado, con o sin retraso), `CANCELED`, `ADDED`… Este es el campo fiable para detectar cancelaciones, no el prefijo `TUCANCEL_` del `id`. Eso responde a una de las preguntas del paso 2. |
| `stop_time_update.stop_id` | La estación a la que se refiere la previsión. Se puede saber cual es al cruzar con la info de `stops.txt` |
| `stop_time_update.arrival.delay` | Retraso previsto en esa estación, en segundos. |
| `stop_time_update.arrival.time` | Hora prevista de llegada a esa estación (Unix UTC). Aquí `1794941460` son las 19:51 en Madrid: unos 45 minutos después de la hora del feed. |
| `trip_update.delay` | Retraso del viaje en conjunto. Está en la especificación como campo experimental y Renfe lo rellena, en este caso con el mismo valor. |
| `vehicle` | Datos del tren. Renfe solo pone si es accesible, sin identificador del vehículo. |

## Entidad con retraso

In [9]:
for entidad in feed.entity:
    viaje = entidad.trip_update
    es_madrid = viaje.trip.trip_id.startswith("10")
    if es_madrid and viaje.delay > 0:
        print(entidad)
        break

id: "TUUPDATE_1020M19722C5"
trip_update {
  trip {
    trip_id: "1020M19722C5"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    arrival {
      delay: 2400
      time: 1794938940
    }
    stop_id: "35606"
  }
  vehicle {
    wheelchair_accessible: WHEELCHAIR_INACCESSIBLE
  }
  delay: 2400
}



## Rutas del proyexto

In [12]:
from pathlib import Path

def raiz_repo(desde: Path = Path.cwd()) -> Path:
    """Sube carpeta a carpeta hasta encontrar la que contiene .git."""
    for carpeta in (desde, *desde.parents):
        if (carpeta / ".git").exists():
            return carpeta
    raise FileNotFoundError(f"No se encontró la raíz del repo desde {desde}")

ROOT = raiz_repo()
DIR_TRIP_UPDATES = ROOT / "data" / "raw" / "gtfs_rt" / "trip_updates"
DIR_TRIP_UPDATES.mkdir(parents=True, exist_ok=True)

print("Raíz:", ROOT)
print("Destino:", DIR_TRIP_UPDATES)

Raíz: e:\Mi Equipo\Evolve\Proyecto_Transversal\Master-Data-Science-IA
Destino: e:\Mi Equipo\Evolve\Proyecto_Transversal\Master-Data-Science-IA\data\raw\gtfs_rt\trip_updates


## Guardamos una captura

In [13]:
hora_captura = datetime.now(timezone.utc)

nombre = f"trip_updates_{hora_captura:%Y%m%dT%H%M%SZ}.pb"
destino = DIR_TRIP_UPDATES / hora_captura.strftime("%Y-%m-%d") / nombre
destino.parent.mkdir(parents=True, exist_ok=True)

destino.write_bytes(response.content)

print("Guardado en:", destino.relative_to(ROOT))
print("Tamaño:", destino.stat().st_size, "bytes")

Guardado en: data\raw\gtfs_rt\trip_updates\2026-11-17\trip_updates_20261117T182959Z.pb
Tamaño: 19804 bytes


## Descargamos y guardamos

In [14]:
import requests

URL_TRIP_UPDATES = "https://gtfsrt.renfe.com/trip_updates.pb"


def capturar_trip_updates() -> dict:
    """Descarga trip_updates, valida que es GTFS-RT y lo guarda tal cual en data/raw."""
    hora_captura = datetime.now(timezone.utc)

    respuesta = requests.get(URL_TRIP_UPDATES, timeout=30)
    respuesta.raise_for_status()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(respuesta.content)  # solo para validar; guardamos los bytes originales

    nombre = f"trip_updates_{hora_captura:%Y%m%dT%H%M%SZ}.pb"
    destino = DIR_TRIP_UPDATES / hora_captura.strftime("%Y-%m-%d") / nombre
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(respuesta.content)

    return {
        "capturado_utc": hora_captura.isoformat(),
        "feed_utc": datetime.fromtimestamp(feed.header.timestamp, tz=timezone.utc).isoformat(),
        "n_entidades": len(feed.entity),
        "bytes": len(respuesta.content),
        "fichero": destino.relative_to(ROOT).as_posix(),
    }


capturar_trip_updates()

{'capturado_utc': '2026-11-17T18:31:03.573725+00:00',
 'feed_utc': '2026-11-17T18:30:44+00:00',
 'n_entidades': 266,
 'bytes': 18939,
 'fichero': 'data/raw/gtfs_rt/trip_updates/2026-11-17/trip_updates_20261117T183103Z.pb'}

### Descarga, Validación y Resumen de la Captura

* **Validación HTTP (`raise_for_status()`):** Lanza una excepción si la respuesta devuelve un código de error HTTP (`404`, `500`, etc.). Sin esta comprobación, un `404` no detendría el flujo y acabaríamos guardando la página HTML de error como si fuera el *feed*.
* **Control de calidad previo al guardado:** Parsear los datos antes de escribirlos en disco actúa como filtro de integridad. Si se recibe contenido que no corresponde a un `FeedMessage` válido, `ParseFromString` lanza un error y evita almacenar datos corruptos o basura.
* **Diccionario de resumen:** El diccionario devuelto sintetiza el resultado de la captura y constituirá la fila que se registrará en el manifiesto en el siguiente paso. Al incluir ambas marcas horarias juntas, cualquier desfase temporal se detecta de un vistazo.
* **Ajuste del *timeout* a 30 s:** Se amplía el margen de espera respecto a los 5 s iniciales, ya que resultaban demasiado ajustados para responder ante la latencia de un servidor público en hora punta.

In [16]:
import csv

MANIFEST = ROOT / "data" / "raw" / "manifest.csv"
COLUMNAS = ["fuente", "capturado_utc", "ok", "error", "feed_utc", "n_entidades", "bytes", "fichero"]


def registrar(fila: dict) -> None:
    """Añade una fila al manifiesto; escribe la cabecera si el fichero es nuevo."""
    es_nuevo = not MANIFEST.exists()
    with MANIFEST.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNAS)
        if es_nuevo:
            writer.writeheader()
        writer.writerow({col: fila.get(col) for col in COLUMNAS})

### Escritura y Estructura del Manifiesto (CSV)

* **Modo `"a"` (*append*):** Añade los registros al final sin sobrescribir lo anterior. De este modo, el manifiesto acumula el histórico completo de todas las ejecuciones (de hoy y de las próximas semanas).
* **Módulo `csv` estándar frente a `pandas`:** Para escribir una única fila no es necesario instanciar un `DataFrame`. Reservaremos `pandas` para la fase posterior de lectura y análisis de datos.
* **Parámetro `newline=""`:** Evita que Windows inserte líneas en blanco adicionales entre filas (un problema clásico al usar el módulo `csv` en este sistema operativo).
* **Esquema de columnas fijas:** Una captura fallida carece de campos como `feed_utc` o `fichero`. Gracias al uso de `fila.get(col)`, esas celdas quedan vacías de forma controlada sin descuadrar la estructura del CSV.
* **Columna `fuente` escalable:** Está diseñada pensando en futuras integraciones (como *alerts*, el GTFS estático o datos meteorológicos), permitiendo centralizar el registro en un único manifiesto para todo el proyecto.

## Captura con gestion de errores

In [17]:
from google.protobuf.message import DecodeError


def capturar_y_registrar() -> dict:
    """Intenta una captura de trip_updates y la apunta en el manifiesto, salga bien o mal."""
    fila = {"fuente": "gtfs_rt/trip_updates"}
    try:
        fila.update(capturar_trip_updates())
        fila["ok"] = True
    except (requests.RequestException, DecodeError) as e:
        fila["capturado_utc"] = datetime.now(timezone.utc).isoformat()
        fila["ok"] = False
        fila["error"] = f"{type(e).__name__}: {e}"
    registrar(fila)
    return fila


capturar_y_registrar()

{'fuente': 'gtfs_rt/trip_updates',
 'capturado_utc': '2026-11-17T18:33:27.970048+00:00',
 'feed_utc': '2026-11-17T18:33:26+00:00',
 'n_entidades': 268,
 'bytes': 19071,
 'fichero': 'data/raw/gtfs_rt/trip_updates/2026-11-17/trip_updates_20261117T183327Z.pb',
 'ok': True}

In [18]:
URL_TRIP_UPDATES = "https://gtfsrt.renfe.com/trip_updatesXX.pb"
print(capturar_y_registrar())
URL_TRIP_UPDATES = "https://gtfsrt.renfe.com/trip_updates.pb"  # la dejamos bien

{'fuente': 'gtfs_rt/trip_updates', 'capturado_utc': '2026-11-17T18:36:07.058548+00:00', 'ok': False, 'error': 'HTTPError: 404 Client Error: Not Found for url: https://gtfsrt.renfe.com/trip_updatesXX.pb'}


### Gestión de Excepciones y Registro de Errores

* **Excepciones capturadas (controladas):**
  * `requests.RequestException`: Agrupa todos los fallos de red (timeouts, falta de conexión, errores HTTP lanzados por `raise_for_status()`, etc.).
  * `DecodeError`: Salta si el archivo descargado no es un formato *protobuf* válido.
* **Excepciones no capturadas:** Cualquier otro error (por ejemplo, un fallo en nuestro propio código) **no se captura** y detiene la ejecución de la celda. Esto es intencionado: usar un `except Exception` genérico ocultaría nuestros propios *bugs* dentro del manifiesto haciéndolos pasar por caídas del servidor de Renfe.
* **Marca de tiempo (*timestamp*) en errores:** En caso de fallo, la hora se registra en este bloque externo, ya que la función interna no llega a devolver su diccionario de resultados.
* **Categorización de fallos:** `type(e).__name__` guarda el nombre exacto de la excepción (`ReadTimeout`, `HTTPError`, etc.), lo que nos permitirá agrupar y analizar los fallos por tipo más adelante.

## Bucle de capturas

In [19]:
import time

N_CAPTURAS = 5
INTERVALO_S = 30

capturas = []
for i in range(N_CAPTURAS):
    inicio = time.monotonic()
    fila = capturar_y_registrar()
    capturas.append(fila)
    print(f"{i + 1}/{N_CAPTURAS}  ok={fila['ok']}  feed={fila.get('feed_utc')}  entidades={fila.get('n_entidades')}")

    if i < N_CAPTURAS - 1:
        transcurrido = time.monotonic() - inicio
        time.sleep(max(0, INTERVALO_S - transcurrido))

1/5  ok=True  feed=2026-11-17T18:37:04+00:00  entidades=263
2/5  ok=True  feed=2026-11-17T18:37:25+00:00  entidades=263
3/5  ok=True  feed=2026-11-17T18:38:04+00:00  entidades=263
4/5  ok=True  feed=2026-11-17T18:38:25+00:00  entidades=262
5/5  ok=True  feed=2026-11-17T18:39:04+00:00  entidades=259


### Bucle de Capturas y Control de Tiempos

* **Parámetros configurables:** Realiza 5 capturas (una cada 30 s, unos 2 minutos en total). Estos valores se definen como constantes al inicio para poder ajustarlos sin modificar la lógica del bucle.
* **Compensación del tiempo de descarga (*drift*):** Descuenta la duración de la petición del tiempo de espera. Si la descarga tarda 3 s, el proceso duerme 27 s en lugar de 30 s, evitando que las capturas se vayan desfasando progresivamente.
* **Medición con `time.monotonic()`:** Es un reloj monotónico que solo avanza hacia adelante, ideal para medir intervalos y duraciones. A diferencia de `datetime.now()`, no se ve afectado si el sistema ajusta el reloj (por ejemplo, en un cambio de horario). Para las marcas de tiempo de los datos seguimos empleando `datetime` en UTC.
* **Evitar esperas innecesarias:** Tras ejecutar la última captura, una condición `if` omite el `sleep` final para no bloquear la celda sin motivo.
* **Almacenamiento en memoria (`capturas`):** Además de escribirse en el CSV, las filas se acumulan en la lista `capturas` para poder inspeccionar únicamente esta tanda en la celda siguiente sin necesidad de leer todo el histórico del manifiesto.

### Decisión de Arquitectura: Frecuencia de Captura del Colector

Estimación de almacenamiento considerando un tamaño promedio de **~19 KB por captura**:

| Intervalo | Capturas / día | Espacio / día | Espacio / mes | Evaluación |
| :--- | :--- | :--- | :--- | :--- |
| **20 s** | 4.320 | ~82 MB | ~2,5 GB | **Sobrecarga innecesaria:** Genera registros redundantes (el *feed* no cambia tan rápido) e infla el almacenamiento sin aportar mayor valor analítico. |
| **60 s** | **1.440** | **~27 MB** | **~820 MB** | **Opción elegida (*Sweet Spot*):** Resolución óptima de 1 minuto para trazar retrasos y cambios de estado con un volumen de datos muy manejable. |
| **5 min** | 288 | ~5,5 MB | ~165 MB | **Granularidad insuficiente:** Se pierde trazabilidad fina en estaciones intermedias y la detección temprana de incidencias puntuales. |

#### Acuerdo del Equipo
TODO: decidir

## Generalizamos para alertas

In [20]:
FEEDS_RT = {
    "trip_updates": "https://gtfsrt.renfe.com/trip_updates.pb",
    "alerts": "https://gtfsrt.renfe.com/alerts.pb",
}
DIR_GTFS_RT = ROOT / "data" / "raw" / "gtfs_rt"


def capturar_feed(nombre: str) -> dict:
    """Descarga un feed GTFS-RT, valida que es un FeedMessage y lo guarda tal cual."""
    hora_captura = datetime.now(timezone.utc)

    respuesta = requests.get(FEEDS_RT[nombre], timeout=30)
    respuesta.raise_for_status()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(respuesta.content)

    destino = (
        DIR_GTFS_RT / nombre / hora_captura.strftime("%Y-%m-%d")
        / f"{nombre}_{hora_captura:%Y%m%dT%H%M%SZ}.pb"
    )
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(respuesta.content)

    return {
        "capturado_utc": hora_captura.isoformat(),
        "feed_utc": datetime.fromtimestamp(feed.header.timestamp, tz=timezone.utc).isoformat(),
        "n_entidades": len(feed.entity),
        "bytes": len(respuesta.content),
        "fichero": destino.relative_to(ROOT).as_posix(),
    }


capturar_feed("alerts")

{'capturado_utc': '2026-11-17T18:41:50.883415+00:00',
 'feed_utc': '2026-11-17T18:41:48+00:00',
 'n_entidades': 85,
 'bytes': 40252,
 'fichero': 'data/raw/gtfs_rt/alerts/2026-11-17/alerts_20261117T184150Z.pb'}

# Actualizamos capturar y registrar que reciba el nombre del feed

In [21]:
def capturar_y_registrar(nombre: str) -> dict:
    """Intenta capturar un feed GTFS-RT y lo apunta en el manifiesto, salga bien o mal."""
    fila = {"fuente": f"gtfs_rt/{nombre}"}
    try:
        fila.update(capturar_feed(nombre))
        fila["ok"] = True
    except (requests.RequestException, DecodeError) as e:
        fila["capturado_utc"] = datetime.now(timezone.utc).isoformat()
        fila["ok"] = False
        fila["error"] = f"{type(e).__name__}: {e}"
    registrar(fila)
    return fila


N_CAPTURAS = 3
INTERVALO_S = 60

capturas = []
for i in range(N_CAPTURAS):
    inicio = time.monotonic()
    for nombre in FEEDS_RT:
        fila = capturar_y_registrar(nombre)
        capturas.append(fila)
        print(f"{i + 1}/{N_CAPTURAS}  {nombre:<13} ok={fila['ok']}  "
              f"feed={fila.get('feed_utc')}  entidades={fila.get('n_entidades')}")

    if i < N_CAPTURAS - 1:
        time.sleep(max(0, INTERVALO_S - (time.monotonic() - inicio)))

1/3  trip_updates  ok=True  feed=2026-11-17T18:43:04+00:00  entidades=263
1/3  alerts        ok=True  feed=2026-11-17T18:43:09+00:00  entidades=85
2/3  trip_updates  ok=True  feed=2026-11-17T18:44:04+00:00  entidades=264
2/3  alerts        ok=True  feed=2026-11-17T18:44:06+00:00  entidades=85
3/3  trip_updates  ok=True  feed=2026-11-17T18:45:04+00:00  entidades=262
3/3  alerts        ok=True  feed=2026-11-17T18:45:06+00:00  entidades=86


### Bucle Multi-feed y Arquitectura de Captura Concurrente

* **Parametrización por nombre:** La función `capturar_y_registrar` acepta el identificador del feed como argumento, empleándolo tanto para realizar la petición HTTP correspondiente como para rellenar la columna `fuente` en el manifiesto.
* **Estructura de bucle anidado:** 
  * El bucle externo controla las **rondas** de captura.
  * El bucle interno recorre los distintos **feeds** configurados (`FEEDS_RT`).
  * En cada ronda se consultan ambos feeds de forma secuencial e inmediata, logrando marcas de tiempo casi idénticas que facilitan cruzar incidencias (por ejemplo, relacionar retrasos con sus avisos correspondientes).
* **Escalabilidad mediante iteración de claves:** El bucle `for nombre in FEEDS_RT` opera directamente sobre las claves del diccionario de configuración; si en el futuro se añade un nuevo feed, el sistema lo integrará automáticamente sin alterar la lógica.
* **Aislamiento de errores por canal:** Un fallo de red o un error al procesar un feed específico **no interrumpe** la captura del otro. Cada feed cuenta con su propio bloque `try` independiente dentro de la función de registro.
* **Configuración temporal:** Se establece el intervalo acordado de **60 segundos**, configurando una prueba de 3 rondas (con una duración total aproximada de 2 minutos).

# Versionar el GTFS estático

In [22]:
import hashlib
import io
import zipfile

URL_GTFS_ESTATICO = "https://ssl.renfe.com/ftransit/Fichero_CER_FOMENTO/fomento_transit.zip"
DIR_GTFS_ESTATICO = ROOT / "data" / "raw" / "gtfs_static"
FICHEROS_OBLIGATORIOS = {"agency.txt", "routes.txt", "trips.txt", "stops.txt", "stop_times.txt"}


def capturar_gtfs_estatico() -> dict:
    """Descarga el GTFS estático; solo guarda una copia nueva si el contenido ha cambiado."""
    hora_captura = datetime.now(timezone.utc)

    respuesta = requests.get(URL_GTFS_ESTATICO, timeout=120)
    respuesta.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(respuesta.content)) as zf:
        faltan = FICHEROS_OBLIGATORIOS - set(zf.namelist())
    if faltan:
        raise ValueError(f"El ZIP no es un GTFS válido, faltan: {sorted(faltan)}")

    huella = hashlib.sha256(respuesta.content).hexdigest()[:12]
    ya_guardado = list(DIR_GTFS_ESTATICO.glob(f"*_{huella}.zip"))

    if ya_guardado:
        destino = ya_guardado[0]
    else:
        destino = DIR_GTFS_ESTATICO / f"fomento_transit_{hora_captura:%Y%m%dT%H%M%SZ}_{huella}.zip"
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(respuesta.content)

    return {
        "capturado_utc": hora_captura.isoformat(),
        "bytes": len(respuesta.content),
        "fichero": destino.relative_to(ROOT).as_posix(),
        "nueva_version": not ya_guardado,
    }


capturar_gtfs_estatico()

{'capturado_utc': '2026-11-17T18:46:32.834908+00:00',
 'bytes': 15621401,
 'fichero': 'data/raw/gtfs_static/fomento_transit_20261117T184632Z_2d29f3dbd394.zip',
 'nueva_version': True}

### Control de Versiones e Integridad del ZIP Estático

* **Huella digital (Hash SHA-256):** Genera un identificador único basado en el contenido del archivo. Un mismo archivo ZIP producirá siempre el mismo hash, mientras que cualquier modificación alterará por completo la cadena. Con utilizar los **primeros 12 caracteres** hay más que suficiente para distinguir versiones de forma unívoca.
* **Estrategia de nombres y control de duplicados:** El hash se incluye directamente en el nombre del archivo. Antes de proceder al guardado, se utiliza `glob` para comprobar si ya existe previamente un fichero con ese hash, evitando duplicar almacenamiento innecesariamente.
* **Marcado temporal en el nombre:** El nombre del archivo incorpora también la fecha, lo que permite identificar con precisión desde cuándo disponemos de cada versión del dataset.
* **Procesamiento en memoria con `io.BytesIO`:** Permite abrir el ZIP directamente en la memoria RAM sin necesidad de escribirlo antes en el disco duro. Su propósito en este punto es únicamente verificar la presencia de los archivos obligatorios de GTFS, sin llegar a descomprimirlo (ya que esa tarea corresponde a una fase posterior de procesamiento).
* **Ampliación del *timeout* a 120 s:** Se incrementa el tiempo límite de espera debido a que el tamaño del archivo ZIP es considerablemente mayor que el de los *feeds* en tiempo real.
* **Variable de control `nueva_version`:** Informa explícitamente del resultado del proceso, indicando si el archivo acaba de ser guardado por primera vez o si ya se encontraba almacenado en el sistema.

# Datos de la metereología

In [23]:
import json

URL_METEO = "https://api.open-meteo.com/v1/forecast"
DIR_METEO = ROOT / "data" / "raw" / "meteo"

PARAMS_METEO = {
    "latitude": 40.4065,   # Madrid-Atocha (aprox.)
    "longitude": -3.6895,
    "hourly": "temperature_2m,precipitation,rain,snowfall,wind_speed_10m,wind_gusts_10m,weather_code",
    "timezone": "Europe/Madrid",
    "past_days": 14,
    "forecast_days": 2,
}


def capturar_meteo() -> dict:
    """Descarga la meteo horaria de Madrid y guarda la respuesta JSON tal cual."""
    hora_captura = datetime.now(timezone.utc)

    respuesta = requests.get(URL_METEO, params=PARAMS_METEO, timeout=30)
    respuesta.raise_for_status()

    datos = respuesta.json()
    if "hourly" not in datos:
        raise ValueError(f"Respuesta sin datos horarios: {list(datos)}")

    destino = DIR_METEO / f"open_meteo_madrid_{hora_captura:%Y%m%dT%H%M%SZ}.json"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(respuesta.content)

    horas = datos["hourly"]["time"]
    return {
        "capturado_utc": hora_captura.isoformat(),
        "n_entidades": len(horas),
        "bytes": len(respuesta.content),
        "fichero": destino.relative_to(ROOT).as_posix(),
        "desde": horas[0],
        "hasta": horas[-1],
    }


capturar_meteo()

{'capturado_utc': '2026-11-17T18:47:35.404746+00:00',
 'n_entidades': 384,
 'bytes': 19790,
 'fichero': 'data/raw/meteo/open_meteo_madrid_20261117T184735Z.json',
 'desde': '2026-11-03T00:00',
 'hasta': '2026-11-18T23:00'}

### Ingesta de Datos Meteorológicos (Open-Meteo)

* **Construcción limpia de URLs (`params=`):** Delegar en el parámetro `params` de `requests` permite construir automáticamente la *query string* (`?latitude=...&hourly=...`) y escapar caracteres especiales de forma segura, evitando concatenaciones manuales propensas a errores.
* **Desfase de zona horaria (`Europe/Madrid` vs. UTC):** La petición solicita las marcas temporales en hora local (`"timezone": "Europe/Madrid"`), que es el formato nativo de respuesta de Open-Meteo. **Nota importante para limpieza:** A diferencia de nuestros *feeds* GTFS (en UTC), estos datos vienen en hora local; será obligatorio normalizar ambas series a una misma zona horaria antes de cruzarlas.
* **Almacenamiento en crudo (`respuesta.content`):** Se guardan directamente los *bytes* originales de la respuesta HTTP en lugar del objeto ya procesado (`datos`). Al igual que con los ficheros *protobuf*, el JSON se parsea en memoria únicamente a modo de validación de integridad.
* **Verificación de cobertura (`desde` y `hasta`):** Permiten confirmar de un vistazo el rango temporal exacto que abarca la respuesta descargada.

#### Limitaciones Conocidas (Ficha de la Fuente)
* **Datos modelizados frente a observaciones reales:** Los registros correspondientes a días pasados no provienen de mediciones físicas de una estación meteorológica, sino de salidas de reanálisis de un modelo numérico. Para obtener observaciones reales habría que recurrir a la API de AEMET (que requiere gestión de clave/API key).
* **Representatividad espacial limitada (punto único):** Todas las consultas toman como referencia las coordenadas de **Atocha** para representar el conjunto de la red. Dado que el núcleo de Cercanías abarca un radio aproximado de 100 km, las condiciones meteorológicas locales (por ejemplo, precipitaciones en Guadalajara) pueden diferir respecto a las registradas en el centro de Madrid.

# Registro común

Revisamos si existe el archivo manifest, ya que se van a añadir campos nuevos y hasta este punto son todo pruebas

In [24]:
import os

archivo = MANIFEST

if os.path.isfile(archivo):
    os.remove(archivo)
    print("El archivo existe. y se ha borrado.")
else:
    print("El archivo no existe.")

El archivo existe. y se ha borrado.


In [25]:
COLUMNAS = ["fuente", "capturado_utc", "ok", "error", "feed_utc", "n_entidades", "bytes", "fichero", "nota"]

CAPTURAS = {
    "gtfs_rt/trip_updates": lambda: capturar_feed("trip_updates"),
    "gtfs_rt/alerts": lambda: capturar_feed("alerts"),
    "gtfs_static": capturar_gtfs_estatico,
    "meteo/open_meteo": capturar_meteo,
}


def capturar_y_registrar(fuente: str) -> dict:
    """Ejecuta la captura de una fuente y la apunta en el manifiesto, salga bien o mal."""
    fila = {"fuente": fuente}
    try:
        resultado = CAPTURAS[fuente]()
        fila.update(resultado)
        fila["ok"] = True
        # Lo que no tiene columna propia se resume en 'nota'
        extras = {k: v for k, v in resultado.items() if k not in COLUMNAS}
        if extras:
            fila["nota"] = "; ".join(f"{k}={v}" for k, v in extras.items())
    except (requests.RequestException, DecodeError, zipfile.BadZipFile, ValueError) as e:
        fila["capturado_utc"] = datetime.now(timezone.utc).isoformat()
        fila["ok"] = False
        fila["error"] = f"{type(e).__name__}: {e}"
    registrar(fila)
    return fila


for fuente in CAPTURAS:
    fila = capturar_y_registrar(fuente)
    print(f"{fuente:<22} ok={fila['ok']}  {fila.get('nota') or fila.get('error') or ''}")

gtfs_rt/trip_updates   ok=True  
gtfs_rt/alerts         ok=True  
gtfs_static            ok=True  nueva_version=False
meteo/open_meteo       ok=True  desde=2026-11-03T00:00; hasta=2026-11-18T23:00


### Orquestador Unificado y Gestión Avanzada de Excepciones

* **Asociación de funciones en el diccionario `CAPTURAS`:** El diccionario mapea cada fuente directamente con su función descargadora correspondiente. Las funciones se almacenan como referencias (sin paréntesis, es decir, sin ejecutarlas en el momento de definirlas) y se invocan de forma dinámica mediante `CAPTURAS[fuente]()`.
* **Funciones anónimas (`lambda`) como adaptadores:** Se emplea una expresión `lambda: capturar_feed("trip_updates")` para empaquetar llamadas que requieren argumentos específicos en una interfaz limpia y sin parámetros. De este modo, todas las entradas del diccionario comparten exactamente la misma firma y pueden ejecutarse de forma homogénea.
* **Columna auxiliar `nueva_nota` para metadatos variables:** Permite almacenar información específica de cada fuente que no dispone de una columna dedicada (por ejemplo, el indicador de `nueva_version` en el GTFS estático o los campos `desde`/`hasta` en la meteo). Esto garantiza que el manifiesto CSV conserve un esquema estricto de columnas fijas.
* **Ampliación de excepciones controladas:** El bloque de captura de errores se enriquece para contemplar incidencias específicas:
  * `BadZipFile`: Salta si el archivo comprimido del GTFS estático llega corrupto o incompleto.
  * `ValueError`: Captura los errores lanzados por nuestras propias validaciones personalizadas (como un ZIP que no contiene los archivos obligatorios de GTFS o una respuesta JSON que carece de la clave `hourly`). *(Nota: `json.JSONDecodeError` hereda directamente de `ValueError`, por lo que también queda cubierto).*

# Resumen del manifiesto

In [27]:
import pandas as pd

manifest = pd.read_csv(MANIFEST)

resumen = manifest.groupby("fuente").agg(
    descargas=("ok", "size"),
    correctas=("ok", "sum"),
    primera=("capturado_utc", "min"),
    ultima=("capturado_utc", "max"),
    megabytes=("bytes", lambda b: round(b.sum() / 1e6, 2)),
)
resumen

,descargas,correctas,primera,ultima,megabytes
fuente,,,,,
gtfs_rt/alerts,1,1,2026-11-17T18:51:50.673360+00:00,2026-11-17T18:51:50.673360+00:00,0.04
gtfs_rt/trip_updates,1,1,2026-11-17T18:51:50.447597+00:00,2026-11-17T18:51:50.447597+00:00,0.02
gtfs_static,1,1,2026-11-17T18:51:50.875569+00:00,2026-11-17T18:51:50.875569+00:00,15.62
meteo/open_meteo,1,1,2026-11-17T18:51:52.234999+00:00,2026-11-17T18:51:52.234999+00:00,0.02


### Agregación y Análisis del Manifiesto con Pandas

* **Uso de `groupby("fuente").agg()`:** Agrupa las filas del manifiesto según la fuente de datos y calcula métricas personalizadas por columna utilizando la sintaxis de tuplas con nombre: `nombre=(columna, función)`.
* **Truco booleano en `correctas=("ok", "sum")`:** Permite calcular el total de ejecuciones exitosas de forma eficiente, ya que `pandas` evalúa los valores booleanos como `1` para `True` (correctas) y `0` para `False` (erróneas).
* **Precaución con la métrica de megabytes:** El cálculo acumulado del GTFS estático suma los `15,6 MB` de cada comprobación cada vez que se registra en el manifiesto, independientemente de que se haya guardado un fichero nuevo en disco o se haya descartado por duplicidad. Es una medida del volumen total descargado por la red, no de la ocupación real del almacenamiento.

### Nota sobre los ficheros de prueba

Durante el desarrollo de este notebook se hicieron capturas de prueba que **están en `data/raw/` pero no en `manifest.csv`**: el manifiesto se recreó al añadir la columna `nota`, y las filas anteriores se perdieron.

Afecta a:
- `data/raw/gtfs_rt/trip_updates/2026-11-17/` y `data/raw/gtfs_rt/alerts/2026-11-17/`: todos los ficheros anteriores a `*_20261117T185150Z.pb`.
- `data/raw/meteo/`: el fichero anterior a `open_meteo_madrid_20261117T185152Z.json`.
- `trip_updates_20261117T182959Z.pb`: **su nombre no corresponde a su contenido**. Contiene el feed de las 18:06:45 UTC, guardado 23 minutos después.

**Criterio para las siguientes fases:** el manifiesto es la fuente de verdad. Solo se procesan los ficheros registrados con `ok=True`. Los ficheros de prueba se conservan como evidencia de la exploración, no como datos del histórico.

# Revision de formato de alerta

In [28]:
# Última captura correcta de alerts, según el manifiesto
manifest = pd.read_csv(MANIFEST)
ultima = manifest[(manifest["fuente"] == "gtfs_rt/alerts") & manifest["ok"]].iloc[-1]

feed_alerts = gtfs_realtime_pb2.FeedMessage()
feed_alerts.ParseFromString((ROOT / ultima["fichero"]).read_bytes())

print("Fichero:", ultima["fichero"])
print("Entidades:", len(feed_alerts.entity))
print(feed_alerts.entity[0])

Fichero: data/raw/gtfs_rt/alerts/2026-11-17/alerts_20261117T185150Z.pb
Entidades: 84
id: "AVISO_526684"
alert {
  active_period {
    start: 1794893400
  }
  informed_entity {
    route_id: "10T0001C1"
  }
  informed_entity {
    route_id: "10T0002C1"
  }
  informed_entity {
    route_id: "10T0005C2"
  }
  informed_entity {
    route_id: "10T0006C2"
  }
  informed_entity {
    route_id: "10T0037C2"
  }
  informed_entity {
    route_id: "10T0007C3"
  }
  informed_entity {
    route_id: "10T0008C3"
  }
  informed_entity {
    route_id: "10T0011C4"
  }
  informed_entity {
    route_id: "10T0012C4"
  }
  informed_entity {
    route_id: "10T0013C4a"
  }
  informed_entity {
    route_id: "10T0014C4a"
  }
  informed_entity {
    route_id: "10T0015C4b"
  }
  informed_entity {
    route_id: "10T0016C4b"
  }
  informed_entity {
    route_id: "10T0017C5"
  }
  informed_entity {
    route_id: "10T0018C5"
  }
  informed_entity {
    route_id: "10T0035C5"
  }
  informed_entity {
    route_id: "10T00

### Puntos de Inspección del Feed de Alertas (`alerts`)

1. **Granularidad de `informed_entity`:** ¿Incluye `route_id`, `stop_id` o `trip`? Es la pregunta clave para determinar a qué nivel de detalle informa Renfe (línea completa, estación concreta o tren específico).
2. **Vigencia en `active_period`:** ¿Define marcas de inicio y fin? Si la fecha de inicio se remonta a hace semanas, indica que se trata de un aviso estructural o permanente (obras, problemas de accesibilidad) y no de una incidencia puntual del momento.
3. **Clasificación en `cause` y `effect`:** ¿Vienen categorizados con valores descriptivos o Renfe los deja por defecto en `UNKNOWN`?
4. **Contenido de `header_text` y `description_text`:** ¿En qué idiomas se publican los mensajes? ¿El texto libre menciona líneas, estaciones o tramos afectados que no aparecen reflejados en los campos estructurados?